# 03 — Barrier Composite Scoring Pipeline

**Purpose:** Produce a scored 400 m × 400 m grid over Barcelona where each cell
receives a composite mycorrhizal-barrier score built from four sub-scores:

| Sub-score | Variable | Proxy for |
|-----------|----------|-----------|
| S1 | Sealed-surface fraction | Physical soil disconnection |
| S2 | LST heat anomaly | Thermal stress on fungal networks |
| S3 | Inverted NDVI | Low canopy → high barrier |
| S4 | Host-mycorrhizal mismatch | Species without fungal partners |

**Input:** `data/grid_trees.geojson` (output of notebook 02)

**Output:** `data/scored_grid.geojson` — all sub-scores, three composite scores,
top-15 flags per scenario, intervention type, colonisation-uncertainty flag,
and sensitivity-warning text.

**Satellite-data fallback:** Urban Atlas, Landsat LST, and Sentinel-2 NDVI rasters
are read from disk when available.  When absent, synthetic values drawn from
ecologically-calibrated distributions are substituted and clearly flagged.
**All synthetic runs must be treated as illustrative only.**

## 0 — Imports and configuration

In [1]:
import rasterio
import numpy as np

with rasterio.open("../data/urban-atlas/sealed_surface.tif") as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    data = src.read(1)
    print("Min:", data.min(), "Max:", data.max(), "Mean:", data.mean().round(4))
    print("Non-zero pixels:", np.count_nonzero(data))
    print("Unique values sample:", np.unique(data)[:10])

CRS: EPSG:25831
Bounds: BoundingBox(left=423600.0, bottom=4577200.0, right=435600.0, top=4591200.0)
Min: 0.024336576 Max: 0.894444 Mean: 0.569
Non-zero pixels: 1050
Unique values sample: [0.02433658 0.0323999  0.03874005 0.06567283 0.0697185  0.07724355
 0.07871895 0.08456304 0.0947432  0.09952685]


In [2]:
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

# rasterio is optional — graceful import
try:
    import rasterio
    from rasterio.mask import mask as rio_mask
    RASTERIO_AVAILABLE = True
except ImportError:
    RASTERIO_AVAILABLE = False
    print("WARNING: rasterio not installed — all raster layers will use synthetic values.")

# scipy for Jaccard and stats
from scipy.stats import beta as beta_dist

np.random.seed(42)
pd.set_option("display.max_columns", 30)

# ── Paths ────────────────────────────────────────────────────────────────────
REPO = Path("../")          # relative to notebooks/
DATA = REPO / "data"

GRID_TREES_PATH      = DATA / "grid_trees.geojson"
SEALED_RASTER_PATH   = DATA / "urban-atlas" / "sealed_surface.tif"
LST_RASTER_PATH      = DATA / "landsat"     / "lst_summer_composite.tif"
NDVI_RASTER_PATH     = DATA / "sentinel2"   / "ndvi_summer_composite.tif"
SCORED_OUTPUT_PATH   = DATA / "scored_grid.geojson"

print("Paths configured.")
for p in [GRID_TREES_PATH, SEALED_RASTER_PATH, LST_RASTER_PATH, NDVI_RASTER_PATH]:
    status = "FOUND" if p.exists() else "ABSENT (will synthesise)"
    print(f"  {p.relative_to(REPO)}  →  {status}")

Paths configured.
  data\grid_trees.geojson  →  FOUND
  data\urban-atlas\sealed_surface.tif  →  FOUND
  data\landsat\lst_summer_composite.tif  →  FOUND
  data\sentinel2\ndvi_summer_composite.tif  →  FOUND


## 1 — Load grid_trees.geojson

If the upstream notebook 02 has not yet been run, we construct a minimal
synthetic grid that mirrors the schema so this notebook remains executable
end-to-end.  All synthesised columns are documented in comments.

In [3]:
def _make_synthetic_grid() -> gpd.GeoDataFrame:
    """Build a representative 400m grid for Barcelona in UTM31N (EPSG:25831)."""
    print("WARNING: grid_trees.geojson not found — generating synthetic grid for demonstration.")
    print("         Run notebook 02 first to obtain real grid data.\n")

    # BCN bounding box in UTM31N (approximate)
    x0, y0, x1, y1 = 419_000, 4_575_000, 436_000, 4_593_000
    cell_size = 400

    xs = np.arange(x0, x1, cell_size)
    ys = np.arange(y0, y1, cell_size)
    xx, yy = np.meshgrid(xs, ys)
    cells = [
        box(x, y, x + cell_size, y + cell_size)
        for x, y in zip(xx.ravel(), yy.ravel())
    ]
    n = len(cells)

    DISTRICTS = [
        "CIUTAT VELLA", "EIXAMPLE", "SANTS - MONTJUÏC", "LES CORTS",
        "SARRIÀ - SANT GERVASI", "GRÀCIA", "HORTA - GUINARDÓ",
        "NOU BARRIS", "SANT ANDREU", "SANT MARTÍ",
    ]
    rng = np.random.default_rng(42)

    gdf = gpd.GeoDataFrame(
        {
            "cell_id":          [f"CELL_{i:04d}" for i in range(n)],
            "nom_districte":    rng.choice(DISTRICTS, size=n),
            "tree_count":       rng.integers(0, 80, size=n),
            "am_pct":           rng.beta(5, 1.5, size=n).clip(0, 1),   # AM dominant in BCN
            "em_pct":           rng.beta(1.5, 5, size=n).clip(0, 1),
            "trees_young_pct":  rng.beta(2, 5, size=n).clip(0, 1),
            # GBIF EM records flag (simulated — 1 = at least one EM obs within 200m)
            "em_gbif_nearby":   rng.choice([0, 1], size=n, p=[0.7, 0.3]),
            "geometry":         cells,
        },
        crs="EPSG:25831",
    )
    return gdf


if GRID_TREES_PATH.exists():
    grid = gpd.read_file(GRID_TREES_PATH)
    print(f"Loaded grid_trees.geojson — {len(grid):,} cells, CRS: {grid.crs}")
else:
    grid = _make_synthetic_grid()
    print(f"Synthetic grid: {len(grid):,} cells, CRS: {grid.crs}")

# Ensure UTM31N projection (required for distance calculations)
if grid.crs is None or grid.crs.to_epsg() != 25831:
    grid = grid.to_crs(epsg=25831)
    print("Re-projected to EPSG:25831 (UTM31N).")

# Normalise required columns if absent (defensive)
for col, default in [("am_pct", 0.8), ("em_pct", 0.1),
                      ("trees_young_pct", 0.2), ("em_gbif_nearby", 0)]:
    if col not in grid.columns:
        grid[col] = default
        print(f"  Column '{col}' missing — filled with {default}")

grid["cell_id"] = grid.get("cell_id", pd.Series([f"CELL_{i:04d}" for i in range(len(grid))]))
print(f"\nGrid shape: {grid.shape}")
print(grid.head(3))

Loaded grid_trees.geojson — 495 cells, CRS: EPSG:25831
  Column 'em_gbif_nearby' missing — filled with 0

Grid shape: (495, 15)
    cell_id  tree_count                                       species_list  \
0  C007_031          11  [Cupressus sempervirens, Ficus carica, Quercus...   
1  C008_031          21  [Celtis australis, Platanus × acerifolia, Plat...   
2  C009_027           3                       [Ficus carica, Quercus ilex]   

           district_name                             barri_name  \
0  SARRIÀ - SANT GERVASI  VALLVIDRERA, EL TIBIDABO I LES PLANES   
1  SARRIÀ - SANT GERVASI  VALLVIDRERA, EL TIBIDABO I LES PLANES   
2  SARRIÀ - SANT GERVASI  VALLVIDRERA, EL TIBIDABO I LES PLANES   

  dominant_myco_type  am_pct  em_pct  nm_pct  am_blindness_flag  \
0                 EM    9.09   45.45   45.45              False   
1                 AM   80.95    0.00   19.05               True   
2                 EM   33.33   66.67    0.00              False   

   trees_young_pct  c

## 2 — Sub-score 1: Sealed-surface fraction (S1)

**Source:** Urban Atlas `sealed_surface.tif` (raster, values 0–100 = % impervious).

**Fallback:** Beta(α=2, β=5) synthetic values — skewed toward low values
(most of Barcelona is sealed but the distribution spans the full range).
The beta parameterisation gives mean ≈ 0.29, mode ≈ 0.17, which reflects
BCN's actual ~30% average sealed fraction at neighbourhood scale.

> **Interpretation:** `sealed_pct` = 1 means fully impervious; higher values
> represent stronger soil-disconnection barriers for mycelial networks.

In [4]:
def zonal_mean_from_raster(raster_path: Path, gdf: gpd.GeoDataFrame,
                            band: int = 1, scale: float = 1.0) -> np.ndarray:
    """
    Compute per-cell zonal mean of a raster band.
    Returns array of length len(gdf); NaN for cells with no valid pixels.
    """
    values = np.full(len(gdf), np.nan, dtype=np.float64)
    with rasterio.open(raster_path) as src:
        # Reproject grid to raster CRS if needed
        gdf_r = gdf if gdf.crs == src.crs else gdf.to_crs(src.crs)
        for i, geom in enumerate(gdf_r.geometry):
            try:
                out_image, _ = rio_mask(src, [geom], crop=True, nodata=src.nodata)
                data = out_image[band - 1].astype(np.float64)
                if src.nodata is not None:
                    data = data[data != src.nodata]
                if data.size > 0:
                    values[i] = np.nanmean(data) * scale
            except Exception:
                pass  # cell outside raster extent — stays NaN
    return values


# ── S1: sealed_pct ─────────────────────────────────────────────────────────
USE_SYNTHETIC_SEALED = True

if RASTERIO_AVAILABLE and SEALED_RASTER_PATH.exists():
    USE_SYNTHETIC_SEALED = False
    print("Reading sealed surface raster…")
    # Urban Atlas raster is already on a 0–1 scale (process_urban_atlas.py).
    # BUG-3 fix: process_urban_atlas.py writes sealed_surface.tif on a 0–1 scale
    # already (verified min=0.024, max=0.894). Do NOT divide by 100 again.
    sealed_raw = zonal_mean_from_raster(SEALED_RASTER_PATH, grid, band=1, scale=1.0)
    grid["sealed_pct"] = np.where(np.isnan(sealed_raw), 0.0, sealed_raw)
    print(f"sealed_pct  min={grid['sealed_pct'].min():.3f}  "
          f"mean={grid['sealed_pct'].mean():.3f}  max={grid['sealed_pct'].max():.3f}")
else:
    print("WARNING: sealed_surface.tif absent — using synthetic Beta(alpha=2, beta=5) values.")
    print("         These are illustrative only and do NOT represent real Urban Atlas data.")
    rng = np.random.default_rng(42)
    grid["sealed_pct"] = rng.beta(2, 5, size=len(grid))   # mean≈0.29, skewed low

# sealed_pct IS the sub-score (higher seal = higher barrier)
grid["s1_sealed"] = grid["sealed_pct"].clip(0, 1)

print(f"\nS1 sealed score — mean: {grid['s1_sealed'].mean():.3f}, "
      f"std: {grid['s1_sealed'].std():.3f}")
print(f"Synthetic values in use: {USE_SYNTHETIC_SEALED}")

Reading sealed surface raster…
sealed_pct  min=0.024  mean=0.646  max=0.894

S1 sealed score — mean: 0.646, std: 0.233
Synthetic values in use: False


## 3 — Sub-score 2: LST heat anomaly (S2)

**Source:** Landsat-8/9 Band 10 summer composite `lst_summer_composite.tif`
(land surface temperature in °C, clipped to BCN).

**Fallback:** Normal(μ=0, σ=2.5) anomaly clipped to [−5, 8] °C — centred on the
city mean, with the upper tail representing heat-island hotspots.

**Normalisation:** min–max across all cells → `lst_score` ∈ [0, 1].  Higher
values = hotter anomaly = stronger thermal barrier.

In [5]:
USE_SYNTHETIC_LST = True

if RASTERIO_AVAILABLE and LST_RASTER_PATH.exists():
    USE_SYNTHETIC_LST = False
    print("Reading LST raster…")
    lst_raw = zonal_mean_from_raster(LST_RASTER_PATH, grid, band=1)
    # Convert to anomaly relative to city median
    city_median = np.nanmedian(lst_raw[~np.isnan(lst_raw)])
    lst_anomaly = lst_raw - city_median
    grid["lst_anomaly_celsius"] = np.where(np.isnan(lst_anomaly), 0.0, lst_anomaly)
else:
    print("WARNING: lst_summer_composite.tif absent — using synthetic N(0, 2.5) anomaly values.")
    print("         Clipped to [-5, 8] °C. These are illustrative only.")
    rng = np.random.default_rng(43)
    anomaly = rng.normal(loc=0, scale=2.5, size=len(grid))
    grid["lst_anomaly_celsius"] = np.clip(anomaly, -5, 8)

# Normalise min–max over all cells
city_min = grid["lst_anomaly_celsius"].min()
city_max = grid["lst_anomaly_celsius"].max()
denom = city_max - city_min
if denom < 1e-9:
    grid["lst_score"] = 0.5
else:
    grid["lst_score"] = (grid["lst_anomaly_celsius"] - city_min) / denom

grid["s2_lst"] = grid["lst_score"].clip(0, 1)

print(f"\nLST anomaly — city_min: {city_min:.2f} °C, city_max: {city_max:.2f} °C")
print(f"S2 LST score — mean: {grid['s2_lst'].mean():.3f}, std: {grid['s2_lst'].std():.3f}")
print(f"Synthetic values in use: {USE_SYNTHETIC_LST}")

Reading LST raster…

LST anomaly — city_min: -9.04 °C, city_max: 7.92 °C
S2 LST score — mean: 0.524, std: 0.138
Synthetic values in use: False


## 4 — Sub-score 3: NDVI canopy (S3)

**Source:** Sentinel-2 Level-2A summer composite `ndvi_summer_composite.tif`
(NDVI, float32, values −1 to 1; well-vegetated cells ≈ 0.4–0.7 in BCN parks).

**Fallback:** Beta(α=3, β=3) rescaled to [0.1, 0.7] — symmetrical on the
vegetation fraction, covering the span from sealed asphalt (~0.1) to
park canopy (~0.7).

**Inversion:** `ndvi_score = 1 − normalised_ndvi` — low canopy means high
mycorrhizal barrier (no trees = no inoculum source).

In [6]:
USE_SYNTHETIC_NDVI = True

if RASTERIO_AVAILABLE and NDVI_RASTER_PATH.exists():
    USE_SYNTHETIC_NDVI = False
    print("Reading NDVI raster…")
    ndvi_raw = zonal_mean_from_raster(NDVI_RASTER_PATH, grid, band=1)
    grid["mean_ndvi"] = np.where(np.isnan(ndvi_raw), 0.3, ndvi_raw)
else:
    print("WARNING: ndvi_summer_composite.tif absent — using synthetic Beta(3,3) on [0.1, 0.7].")
    print("         These are illustrative only.")
    rng = np.random.default_rng(44)
    raw = rng.beta(3, 3, size=len(grid))   # symmetric, mean=0.5
    grid["mean_ndvi"] = 0.1 + raw * (0.7 - 0.1)   # rescale to [0.1, 0.7]

# Normalise mean_ndvi to [0,1] then invert
ndvi_min = grid["mean_ndvi"].min()
ndvi_max = grid["mean_ndvi"].max()
ndvi_range = ndvi_max - ndvi_min
if ndvi_range < 1e-9:
    normalised_ndvi = 0.5
else:
    normalised_ndvi = (grid["mean_ndvi"] - ndvi_min) / ndvi_range

grid["s3_ndvi"] = (1 - normalised_ndvi).clip(0, 1)

print(f"\nNDVI range: [{grid['mean_ndvi'].min():.3f}, {grid['mean_ndvi'].max():.3f}]")
print(f"S3 NDVI barrier score — mean: {grid['s3_ndvi'].mean():.3f}, "
      f"std: {grid['s3_ndvi'].std():.3f}")
print(f"Synthetic values in use: {USE_SYNTHETIC_NDVI}")

Reading NDVI raster…

NDVI range: [-0.027, 0.364]
S3 NDVI barrier score — mean: 0.590, std: 0.163
Synthetic values in use: False


## 5 — Sub-score 4: Host-mycorrhizal mismatch (S4)

**Weight rationale (0.05):** The AM-blindness problem identified in notebook 01
means ~80–95% of Barcelona trees (Platanus, Celtis, Tipuana…) are AM hosts
whose fungal partners produce no aboveground fruiting bodies visible to
citizen-science platforms like GBIF iNaturalist.  For AM-dominant cells the
mismatch score is therefore informationally null — we have no signal, not
necessarily no mismatch.  Assigning this a 0.05 weight prevents it from
driving rankings while preserving the EM-host signal where it exists.

**Scoring rules:**

| Condition | `mismatch_score` | Rationale |
|-----------|------------------|-----------|
| AM-dominant (am_pct ≥ 0.8) | 0.5 | Informationally null — no signal |
| EM-dominant (em_pct ≥ 0.5) + GBIF EM records nearby | 0.0 | Partners present |
| EM-dominant (em_pct ≥ 0.5) + no GBIF EM records | 0.8 | Potential isolation |
| Mixed | 0.6 | Uncertain — moderate concern |

In [7]:
def compute_mismatch_score(am_pct: pd.Series, em_pct: pd.Series,
                            em_gbif_nearby: pd.Series) -> pd.Series:
    """
    Rule-based mismatch score.
    em_gbif_nearby: 1 = at least one confirmed EM observation within 200 m,
                    0 = no observation (could be absence or observer gap).
    """
    score = pd.Series(np.full(len(am_pct), 0.6), index=am_pct.index)   # default: mixed

    # EM-dominant: score depends on GBIF evidence
    em_dom = em_pct >= 50  # BUG-4 fix: am_pct/em_pct stored on 0-100 scale
    score[em_dom & (em_gbif_nearby == 1)] = 0.0   # partners documented
    score[em_dom & (em_gbif_nearby == 0)] = 0.8   # potential isolation

    # AM-dominant overrides (applied last so AM takes precedence over mixed)
    am_dom = am_pct >= 80  # BUG-4 fix: 0-100 scale
    score[am_dom] = 0.5   # informationally null

    return score


grid["s4_mismatch"] = compute_mismatch_score(
    grid["am_pct"], grid["em_pct"], grid["em_gbif_nearby"]
)

print("S4 mismatch score distribution:")
print(grid["s4_mismatch"].value_counts().sort_index().to_frame("cell_count").to_string())
print(f"\nAM-dominant cells (score=0.5): {(grid['am_pct'] >= 80).sum():,}")
print(f"EM-dominant cells:             {(grid['em_pct'] >= 50).sum():,}")
print(f"Mixed cells (score=0.6):       "
      f"{((grid['am_pct'] < 80) & (grid['em_pct'] < 50)).sum():,}")

S4 mismatch score distribution:
             cell_count
s4_mismatch            
0.5                 263
0.6                 216
0.8                  16

AM-dominant cells (score=0.5): 263
EM-dominant cells:             16
Mixed cells (score=0.6):       216


## 6 — Composite scoring under three weight scenarios

Three scenarios test whether the top-15 priority zones are robust to
different disciplinary emphases.

| Scenario | S1 sealed | S2 LST | S3 NDVI | S4 mismatch | Rationale |
|----------|-----------|--------|---------|-------------|-----------|
| A — equal | 0.25 | 0.25 | 0.25 | 0.25 | No prior assumptions |
| B — sealed dominant | 0.50 | 0.17 | 0.17 | 0.05 (↓ AM-blind) | Physical barrier is primary |
| C — heat + canopy | 0.17 | 0.30 | 0.30 | 0.23 | Climate-adaptation emphasis |

In [8]:
SCENARIOS = {
    "A": {"sealed": 0.25, "lst": 0.25, "ndvi": 0.25, "mismatch": 0.25},
    "B": {"sealed": 0.55, "lst": 0.2, "ndvi": 0.2, "mismatch": 0.05},
    "C": {"sealed": 0.17, "lst": 0.30, "ndvi": 0.30, "mismatch": 0.23},
}

for label, w in SCENARIOS.items():
    total_w = sum(w.values())
    assert abs(total_w - 1.0) < 1e-6, (
        f"Scenario {label} weights sum to {total_w:.4f}, not 1.0"
    )
    grid[f"composite_{label}"] = (
        w["sealed"]   * grid["s1_sealed"]  +
        w["lst"]      * grid["s2_lst"]     +
        w["ndvi"]     * grid["s3_ndvi"]    +
        w["mismatch"] * grid["s4_mismatch"]
    )
    grid[f"rank_{label}"] = grid[f"composite_{label}"].rank(
        ascending=False, method="first"
    ).astype(int)
    print(f"Scenario {label}: composite range "
          f"[{grid[f'composite_{label}'].min():.3f}, "
          f"{grid[f'composite_{label}'].max():.3f}]")

print("\nComposite scores computed for all three scenarios.")

Scenario A: composite range [0.207, 0.812]
Scenario B: composite range [0.093, 0.855]
Scenario C: composite range [0.196, 0.835]

Composite scores computed for all three scenarios.


## 7 — Top-15 selection with district-representation constraint

A naive top-15 list risks concentrating on a single well-scored district.
We apply a **district-guarantee constraint:** every district with at least
one high-scoring cell must have ≥ 1 representative in the top-15.  If the
15th-ranked cell must be displaced to accommodate an unrepresented district,
it is replaced by the highest-ranked cell from that district.

In [9]:
def select_top15_with_district_constraint(
    gdf: gpd.GeoDataFrame,
    composite_col: str,
    district_col: str = "nom_districte",
    k: int = 15,
) -> pd.Index:
    """
    Select top-k cells ranked by composite_col.
    Ensure every district that has any cells in the full grid has at least
    one representative in the returned set.

    BUG-6 fix: previous version overwrote selected[-1] for every missing
    district, so only the LAST missing district got a representative. The
    new logic adds one cell per missing district, displacing the lowest-
    ranked currently-selected cell whose district has > 1 representative.

    Returns: pandas Index of selected cell index labels.
    """
    scored = gdf.sort_values(composite_col, ascending=False)
    selected = scored.head(k).copy()
    selected_districts = set(selected[district_col].dropna().unique())
    all_districts = set(scored[district_col].dropna().unique())
    missing = sorted(all_districts - selected_districts)

    for d in missing:
        candidate = scored[scored[district_col] == d].nlargest(1, composite_col)
        if candidate.empty:
            continue
        # Find the lowest-ranked currently-selected cell whose district has
        # more than one representative — that's the safe one to drop.
        selected_sorted = selected.sort_values(composite_col)
        drop_idx = None
        for idx in selected_sorted.index:
            cell_d = selected.loc[idx, district_col]
            if (selected[district_col] == cell_d).sum() > 1:
                drop_idx = idx
                break
        if drop_idx is not None:
            selected = selected.drop(drop_idx)
        # If no over-represented district exists we simply grow the set —
        # the constraint is "at least one per district", so growing is fine.
        selected = pd.concat([selected, candidate])
    selected = selected.sort_values(composite_col, ascending=False)
    return selected.index

if "district_name" in grid.columns and "nom_districte" not in grid.columns:
      grid = grid.rename(columns={"district_name": "nom_districte"})
top15_indices = {}
for label in ["A", "B", "C"]:
    idx = select_top15_with_district_constraint(
        grid, f"composite_{label}", district_col="nom_districte", k=15
    )
    top15_indices[label] = idx
    col = f"top15_scenario_{label}"
    grid[col] = False
    grid.loc[idx, col] = True
    n_districts = grid.loc[idx, "nom_districte"].nunique()
    print(f"Scenario {label}: {len(idx)} cells selected, {n_districts} districts represented")

print("\nTop-15 district coverage (Scenario B — primary):")
print(
    grid[grid["top15_scenario_B"]][["nom_districte", "composite_B", "rank_B"]]
    .sort_values("rank_B")
    .to_string(index=False)
)

Scenario A: 15 cells selected, 10 districts represented
Scenario B: 15 cells selected, 10 districts represented
Scenario C: 15 cells selected, 10 districts represented

Top-15 district coverage (Scenario B — primary):
        nom_districte  composite_B  rank_B
     SANTS - MONTJUÏC     0.855370       1
          SANT ANDREU     0.820688       2
          SANT ANDREU     0.817259       3
     SANTS - MONTJUÏC     0.797454       4
     SANTS - MONTJUÏC     0.796826       5
     SANTS - MONTJUÏC     0.794314       6
         CIUTAT VELLA     0.788289       7
     SANTS - MONTJUÏC     0.787718       8
           SANT MARTÍ     0.786323       9
     HORTA - GUINARDÓ     0.775434      15
SARRIÀ - SANT GERVASI     0.765726      33
             EIXAMPLE     0.764214      35
            LES CORTS     0.760931      42
           NOU BARRIS     0.752664      65
               GRÀCIA     0.746666      87


## 8 — Sensitivity check: Jaccard similarity between top-15 sets

Jaccard(A, B) = |A ∩ B| / |A ∪ B|.  A pair with Jaccard < 0.5 indicates
the rankings are weight-sensitive and no single scenario should be presented
as definitive.

In [10]:
def jaccard(set_a: set, set_b: set) -> float:
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


sets = {k: set(v.tolist()) for k, v in top15_indices.items()}
pairs = [("A", "B"), ("A", "C"), ("B", "C")]
jaccard_scores = {}

print("Jaccard similarity between top-15 sets:")
for a, b in pairs:
    j = jaccard(sets[a], sets[b])
    jaccard_scores[(a, b)] = j
    flag = " *** WEIGHT-SENSITIVE" if j < 0.5 else ""
    print(f"  Scenarios {a} vs {b}: {j:.3f}{flag}")

weight_sensitive = any(j < 0.5 for j in jaccard_scores.values())

if weight_sensitive:
    SENSITIVITY_WARNING = (
        "Rankings are weight-sensitive — present all 3 scenarios, "
        "do not present any single scenario as definitive."
    )
    print(f"\nWARNING: {SENSITIVITY_WARNING}")
else:
    SENSITIVITY_WARNING = (
        "Rankings are weight-robust — Scenario B recommended as primary."
    )
    print(f"\n{SENSITIVITY_WARNING}")

grid["sensitivity_warning"] = SENSITIVITY_WARNING

# Also store Jaccard scores for reference
grid["jaccard_AB"] = round(jaccard_scores[("A", "B")], 4)
grid["jaccard_AC"] = round(jaccard_scores[("A", "C")], 4)
grid["jaccard_BC"] = round(jaccard_scores[("B", "C")], 4)

Jaccard similarity between top-15 sets:
  Scenarios A vs B: 0.364 *** WEIGHT-SENSITIVE
  Scenarios A vs C: 0.875
  Scenarios B vs C: 0.364 *** WEIGHT-SENSITIVE



## 9 — Intervention type classification

Using Scenario B as primary, the dominant sub-score determines the
intervention recommendation per cell:

| Dominant sub-score | Intervention type |
|--------------------|-------------------|
| S1 sealed | `de-paving` |
| S2 LST | `cooling` |
| S3 NDVI | `planting` |
| S4 mismatch | `species-selection` |

In [11]:
# --------------------------------------------------------------
# Intervention profile (Fix 3, 2026-05-10)
# --------------------------------------------------------------
# Per the Geographer's review (outputs/geographer-review.md, section 4),
# single intervention labels are geographically incoherent: cells like
# La Marina del Port score high on sealing AND temperature AND
# canopy-loss simultaneously. Labelling them "de-paving" alone strips
# planners of the compound context they need and risks producing
# unshaded de-paved scars instead of an integrated intervention.
#
# This cell replaces the single-label argmax with a *profile vector*
# that records each sub-score's percentage contribution to composite_B.
# - intervention_profile      : dict, e.g. {"de-paving": 60.2, ...}
# - intervention_type         : single dominant label (back-compat)
# - intervention_profile_str  : human-readable, e.g.
#                               "60% de-paving · 30% cooling · 10% planting"

W_B = SCENARIOS["B"]
SCENARIO_B_WEIGHTS = {
    "s1_sealed":   W_B["sealed"],
    "s2_lst":      W_B["lst"],
    "s3_ndvi":     W_B["ndvi"],
    "s4_mismatch": W_B["mismatch"],
}
LABEL_MAP = {
    "s1_sealed":   "de-paving",
    "s2_lst":      "cooling",
    "s3_ndvi":     "planting",
    "s4_mismatch": "species-selection",
}


def _intervention_profile(row):
    """Per-cell percentage contribution of each sub-score to composite_B.

    Returns a dict {label: percentage}. Percentages sum to ~100 unless
    the weighted composite for this cell is zero, in which case every
    label is 0.
    """
    contributions = {LABEL_MAP[k]: row[k] * w for k, w in SCENARIO_B_WEIGHTS.items()}
    total = sum(contributions.values())
    if total <= 0:
        return {label: 0.0 for label in LABEL_MAP.values()}
    return {label: round(v / total * 100.0, 1) for label, v in contributions.items()}


grid["intervention_profile"] = grid.apply(_intervention_profile, axis=1)

# Dominant intervention — kept for back-compat with downstream consumers.
grid["intervention_type"] = grid["intervention_profile"].apply(
    lambda p: max(p.items(), key=lambda x: x[1])[0]
)


def _profile_str(p):
    """Compact human-readable, dropping shares < 5%."""
    sorted_items = sorted(p.items(), key=lambda x: -x[1])
    return " · ".join(f"{v:.0f}% {k}" for k, v in sorted_items if v >= 5)


grid["intervention_profile_str"] = grid["intervention_profile"].apply(_profile_str)

# --- diagnostics ---
print("Dominant intervention — all cells:")
print(grid["intervention_type"].value_counts().to_string())
print("\nDominant intervention — top-15 Scenario B:")
print(
    grid[grid["top15_scenario_B"]]["intervention_type"]
    .value_counts().to_string()
)
print("\nProfile strings — top-15 Scenario B (sorted by composite_B):")
for _, r in (
    grid[grid["top15_scenario_B"]]
    .sort_values("composite_B", ascending=False)
    .iterrows()
):
    print(f"  {r['cell_id']:<10s}  {r['intervention_profile_str']}")


Dominant intervention — all cells:
intervention_type
de-paving            471
planting              14
cooling                9
species-selection      1

Dominant intervention — top-15 Scenario B:
intervention_type
de-paving    15

Profile strings — top-15 Scenario B (sorted by composite_B):
  C016_011    52% de-paving · 23% cooling · 22% planting
  C031_035    54% de-paving · 22% cooling · 22% planting
  C032_032    54% de-paving · 22% planting · 21% cooling
  C020_016    56% de-paving · 23% planting · 17% cooling
  C016_010    56% de-paving · 21% planting · 20% cooling
  C014_016    61% de-paving · 19% planting · 16% cooling
  C026_019    62% de-paving · 21% planting · 14% cooling
  C015_016    62% de-paving · 19% planting · 16% cooling
  C027_027    62% de-paving · 19% planting · 16% cooling
  C025_030    62% de-paving · 19% planting · 15% cooling
  C020_024    64% de-paving · 19% planting · 13% cooling
  C025_020    64% de-paving · 19% planting · 13% cooling
  C015_019    62% de-pa

## 10 — Colonisation uncertainty flag

Cells in the top-15 where ≥ 30% of trees were planted recently
(`trees_young_pct ≥ 0.3`) receive a `colonisation_uncertain = True` flag.
Young trees may lack established mycorrhizal associations — any
connectivity gain from de-paving may take 5–10 seasons to materialise.

In [12]:
grid["colonisation_uncertain"] = (
    grid["top15_scenario_B"] & (grid["trees_young_pct"] >= 30)  # BUG-5 fix: 0-100 scale
)

n_uncertain = grid["colonisation_uncertain"].sum()
print(f"Top-15 (Scenario B) cells with colonisation uncertainty flag: {n_uncertain}")
if n_uncertain > 0:
    print()
    print(
        grid[grid["colonisation_uncertain"]][
            ["cell_id", "nom_districte", "trees_young_pct", "composite_B"]
        ].to_string(index=False)
    )

Top-15 (Scenario B) cells with colonisation uncertainty flag: 0


## 11 — Summary: top-15 priority zones (Scenario B, primary)

Full ranked table with all sub-scores and flags.

In [13]:
DISPLAY_COLS = [
    "cell_id", "nom_districte",
    "s1_sealed", "s2_lst", "s3_ndvi", "s4_mismatch",
    "composite_B", "rank_B",
    "top15_scenario_A", "top15_scenario_B", "top15_scenario_C",
    "intervention_type", "colonisation_uncertain",
]

available_cols = [c for c in DISPLAY_COLS if c in grid.columns]
top15_B = (
    grid[grid["top15_scenario_B"]][available_cols]
    .sort_values("rank_B")
    .reset_index(drop=True)
)
top15_B.index = top15_B.index + 1   # 1-based rank

pd.set_option("display.float_format", "{:.3f}".format)
print(f"Top-15 Priority Zones — Scenario B (Sealed-dominant weights)\n")
print(top15_B.to_string())
print(f"\nSensitivity: {SENSITIVITY_WARNING}")

Top-15 Priority Zones — Scenario B (Sealed-dominant weights)

     cell_id          nom_districte  s1_sealed  s2_lst  s3_ndvi  s4_mismatch  composite_B  rank_B  top15_scenario_A  top15_scenario_B  top15_scenario_C intervention_type  colonisation_uncertain
1   C016_011       SANTS - MONTJUÏC      0.802   1.000    0.946        0.500        0.855       1              True              True              True         de-paving                   False
2   C031_035            SANT ANDREU      0.802   0.890    0.883        0.500        0.821       2              True              True              True         de-paving                   False
3   C032_032            SANT ANDREU      0.806   0.859    0.886        0.500        0.817       3              True              True              True         de-paving                   False
4   C020_016       SANTS - MONTJUÏC      0.818   0.674    0.914        0.600        0.797       4              True              True              True         de

## 12 — Save scored_grid.geojson

In [14]:
# Drop intermediate / temporary columns that shouldn't appear in the output
drop_cols = ["lst_score"]   # already captured as s2_lst
output_grid = grid.drop(columns=[c for c in drop_cols if c in grid.columns])

# Ensure boolean columns are saved as true booleans
for col in ["top15_scenario_A", "top15_scenario_B", "top15_scenario_C",
             "colonisation_uncertain"]:
    if col in output_grid.columns:
        output_grid[col] = output_grid[col].astype(bool)

output_grid.to_file(SCORED_OUTPUT_PATH, driver="GeoJSON")

print(f"Saved: {SCORED_OUTPUT_PATH}")
print(f"  Rows: {len(output_grid):,}")
print(f"  Columns: {list(output_grid.columns)}")
print(f"  CRS: {output_grid.crs}")
print(f"  File size: {SCORED_OUTPUT_PATH.stat().st_size / 1024:.1f} KB")
print()
print("Synthetic data flags:")
print(f"  sealed_pct    synthetic: {USE_SYNTHETIC_SEALED}")
print(f"  lst_anomaly   synthetic: {USE_SYNTHETIC_LST}")
print(f"  mean_ndvi     synthetic: {USE_SYNTHETIC_NDVI}")

Saved: ..\data\scored_grid.geojson
  Rows: 495
  Columns: ['cell_id', 'tree_count', 'species_list', 'nom_districte', 'barri_name', 'dominant_myco_type', 'am_pct', 'em_pct', 'nm_pct', 'am_blindness_flag', 'trees_young_pct', 'cell_x0', 'cell_y0', 'geometry', 'em_gbif_nearby', 'sealed_pct', 's1_sealed', 'lst_anomaly_celsius', 's2_lst', 'mean_ndvi', 's3_ndvi', 's4_mismatch', 'composite_A', 'rank_A', 'composite_B', 'rank_B', 'composite_C', 'rank_C', 'top15_scenario_A', 'top15_scenario_B', 'top15_scenario_C', 'sensitivity_warning', 'jaccard_AB', 'jaccard_AC', 'jaccard_BC', 'intervention_profile', 'intervention_type', 'intervention_profile_str', 'colonisation_uncertain']
  CRS: EPSG:25831
  File size: 1006.7 KB

Synthetic data flags:
  sealed_pct    synthetic: False
  lst_anomaly   synthetic: False
  mean_ndvi     synthetic: False
